# C1 data build: from sorted running files to the model inputs

This notebook reads the sorted running `.fit` files and builds the two files every model in this project is trained on. It opens each file once.

| | |
|---|---|
| **Input** | the running files in `garmin_data\DI_CONNECT\Sorted\Run` (named `DDMMYYYY__activityId.fit`) |
| **Output 1** | `X_flat_c1_v2.npy`: one row per 30-second window, 121 numbers: heart rate (30), speed (30), cadence (30), temperature (30), distance covered at the start of the window (1) |
| **Output 2** | `y_labels_c1_v2.csv`: one row per window, in the same order as X: `run_idx`, `date`, `file_path`, `start_s`, `end_s`, `c1_target` (the label of the NEXT window) |

## How the data is built
1. **Find the run files** and put them in a fixed order (date, then file name), so `run_idx` is the same every time.
2. **Read each file** (once): heart rate, speed, cadence, temperature, distance and timestamp from every record. Heart rate, speed and timestamp are required, and a file needs at least 50 usable records. The other three fields may be missing.
3. **Clean speed.** Values above 25 km/h or below 0 are treated as errors and removed. Up to 5 removed readings in a row are filled by a straight line, and any further ones take the next valid reading.
4. **Resample** every field to exactly one value per second. Seconds with no record, including any pause, are filled by a straight line between the readings either side, whatever the length.
5. **Cut into windows:** consecutive, non-overlapping 30-second windows.
6. **Label each window** (see below).
7. **Forecast target.** A window's target is the label of the window after it, in the same run.
8. **Extract the model input** for every window that has a target. Missing cadence, temperature or distance become 0.
9. **Sort** the rows by file name, then window start, and save X and Y.

## How a window is labelled
Efficiency is speed divided by heart rate, smoothed over 90 seconds. Its slope (change per minute) over the window decides the label:

| Label | Rule |
|---|---|
| `Recovery` | average heart rate and average speed are both below 85% of the run's averages |
| `Declining` | slope at or below minus 0.0003 per minute |
| `Improving` | slope at or above plus 0.0003 per minute |
| `Base` | anything else |

Two further rules: a `Declining` or `Improving` stretch must last at least 2 consecutive windows, otherwise it becomes `Base`; and the first 120 seconds of every run are warm-up and are left unlabelled.

## 1. Settings

In [1]:
import time
from collections import Counter
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from garmin_fit_sdk import Decoder, Stream

# ---- Paths ----
RUN_DIR = Path(r"C:\dev\UoW\DATAX504\Final Assignment\garmin_data\DI_CONNECT\Sorted\Run")      # input
OUT_DIR = Path(r"C:\dev\UoW\DATAX504\Final Assignment\garmin_data\Processed")            # output: a new folder for the check run

# ---- Settings ----
WINDOW_SECONDS = 30            # length of one window
ROLLING_TREND_WINDOW = 90      # seconds over which efficiency is smoothed
WARMUP_EXCLUDE_SECONDS = 120   # start of each run left unlabelled
FIXED_SLOPE_THRESHOLD = 0.0003 # efficiency change per minute that counts as Declining/Improving
MIN_PERSISTENCE_WINDOWS = 2    # a Declining/Improving stretch must last at least this many windows
RECOVERY_FRACTION = 0.85       # Recovery: HR and speed both below this fraction of the run average
FORECAST_HORIZON = 1           # target = the label this many windows ahead (1 = the next window)
MIN_RECORDS = 50               # minimum usable records in a file
MAX_SPEED_KMH = 25             # faster than this is treated as an error
SPEED_INTERP_LIMIT = 5         # removed speed readings in a row that are filled by a straight line (the rest take the next valid reading)

# ---- Output files ----
OUT_DIR.mkdir(parents=True, exist_ok=True)
X_OUT = OUT_DIR / "X_flat_c1_v2_rebuild.npy"
Y_OUT = OUT_DIR / "y_labels_c1_v2_rebuild.csv"

## 2. Find the run files

File names look like `17042017__11078969721.fit`: the date (day, month, year), then the activity ID.
Files are ordered by date, then by name, so every run gets the same `run_idx`.


In [2]:
def date_from_filename(fpath):
    return datetime.strptime(fpath.stem.split("__")[0], "%d%m%Y")

run_files = list(RUN_DIR.glob("*.fit"))
dated, unreadable_names = [], []
for f in run_files:
    try:
        dated.append((f, date_from_filename(f)))
    except ValueError:
        unreadable_names.append(f.name)
dated.sort(key=lambda item: (item[1], item[0].name))

print(f"{len(run_files)} run files found, from {dated[0][1].date()} to {dated[-1][1].date()}"
      f" ({len(unreadable_names)} with an unreadable date in the name)")


1891 run files found, from 2017-04-17 to 2026-07-13 (0 with an unreadable date in the name)


## 3. Read, clean and resample one run

`read_fit_records` is the only place a `.fit` file is opened. Heart rate, speed
and timestamp are required in a record; cadence, temperature and distance are
kept when present. Speed is cleaned, then every field is resampled to one value per second.


In [3]:
def read_fit_records(fpath):
    """Decode one .fit file; return (DataFrame of per-second-ish records, None) or (None, reason)."""
    stream = Stream.from_file(str(fpath))
    decoder = Decoder(stream)
    messages, errors = decoder.read()

    if "session_mesgs" not in messages or "record_mesgs" not in messages:
        return None, "no session or record messages"

    rows = []
    for record in messages["record_mesgs"]:
        hr = record.get("heart_rate")
        speed = record.get("enhanced_speed", record.get("speed"))
        cadence = record.get("cadence")
        temperature = record.get("temperature")
        distance = record.get("distance")
        timestamp = record.get("timestamp")
        if None not in (hr, speed, timestamp):          # heart rate, speed and timestamp are required
            rows.append({"timestamp": timestamp, "heart_rate": hr, "speed": speed,
                         "cadence": cadence, "temperature": temperature, "distance": distance})

    if len(rows) < MIN_RECORDS:
        return None, f"fewer than {MIN_RECORDS} usable records"

    df = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
    df["elapsed_s"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()
    return df, None


def clean_speed(speed_array, max_kmh=MAX_SPEED_KMH):
    speed_kmh = speed_array * 3.6
    implausible = (speed_kmh > max_kmh) | (speed_kmh < 0)
    cleaned = pd.Series(speed_array.copy())
    cleaned[implausible] = np.nan
    cleaned = cleaned.interpolate(limit=SPEED_INTERP_LIMIT)
    return cleaned.bfill().ffill().to_numpy()


def resample_to_1s(df):
    df = df.drop(columns=["timestamp"])
    for col in ["heart_rate", "speed", "cadence", "temperature", "distance"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    indexed = df.set_index("elapsed_s")
    full_index = np.arange(0, int(indexed.index.max()) + 1, 1)
    resampled = indexed.reindex(indexed.index.union(full_index)).sort_index()
    resampled = resampled.interpolate(method="linear", limit_direction="both")
    resampled = resampled.reindex(full_index)
    resampled["elapsed_s"] = resampled.index
    return resampled.reset_index(drop=True)


## 4. Label the windows

Efficiency (speed divided by heart rate) is smoothed and its slope is taken.
Each 30-second window then gets one label from the rules in the table at the top.


In [4]:
def compute_efficiency_trend(df, window=ROLLING_TREND_WINDOW):
    speed = df["speed"].to_numpy()
    heart_rate = df["heart_rate"].to_numpy()
    efficiency = np.divide(speed, heart_rate, out=np.zeros_like(speed), where=heart_rate > 0)
    rolling_efficiency = pd.Series(efficiency).rolling(window, center=True, min_periods=max(window // 4, 10)).mean().to_numpy()
    slope = np.gradient(rolling_efficiency, df["elapsed_s"].to_numpy())
    return slope


def label_windows(df, slope):
    """One effectiveness label per 30-second window of a run (None during warm-up)."""
    speed = df["speed"].to_numpy()
    heart_rate = df["heart_rate"].to_numpy()
    run_avg_hr, run_avg_speed = heart_rate.mean(), speed.mean()
    starts = list(range(0, len(df) - WINDOW_SECONDS, WINDOW_SECONDS))

    labels = []
    for start in starts:
        end = start + WINDOW_SECONDS
        if start < WARMUP_EXCLUDE_SECONDS:
            labels.append(None)                                   # warm-up: unlabelled
            continue
        avg_hr = heart_rate[start:end].mean()
        avg_speed = speed[start:end].mean()
        slope_per_min = np.nanmean(slope[start:end]) * 60
        if avg_hr < run_avg_hr * RECOVERY_FRACTION and avg_speed < run_avg_speed * RECOVERY_FRACTION:
            labels.append("Recovery")
        elif slope_per_min <= -FIXED_SLOPE_THRESHOLD:
            labels.append("Declining")
        elif slope_per_min >= FIXED_SLOPE_THRESHOLD:
            labels.append("Improving")
        else:
            labels.append("Base")

    # Persistence rule: a Declining/Improving stretch shorter than MIN_PERSISTENCE_WINDOWS becomes Base.
    i = 0
    while i < len(labels):
        if labels[i] in ("Declining", "Improving"):
            j = i
            while j < len(labels) and labels[j] == labels[i]:
                j += 1
            if (j - i) < MIN_PERSISTENCE_WINDOWS:
                labels[i:j] = ["Base"] * (j - i)
            i = j
        else:
            i += 1

    return pd.DataFrame({"start_s": starts, "end_s": [s + WINDOW_SECONDS for s in starts], "label": labels})


## 5. Build one run: windows, targets and model inputs

For every window that has a next window, the function keeps the next window's
label as the target and extracts the 121 input values. Two things can drop
input rows: a file that cannot be read at all is skipped entirely, and a failure
while extracting values drops only that run's input rows (recorded in the manifest).


In [14]:
def process_run(fpath):
    """Everything needed from one run file. Returns a dict"""
    try:
        df, reason = read_fit_records(fpath)
    except Exception as exc:
        return {"status": "skipped", "reason": f"decode error: {type(exc).__name__}: {exc}"}
    if df is None:
        return {"status": "skipped", "reason": reason}

    try:
        df["speed"] = clean_speed(df["speed"].to_numpy())
        df = resample_to_1s(df)
        heart_rate, speed = df["heart_rate"].to_numpy(), df["speed"].to_numpy()

        windows = label_windows(df, compute_efficiency_trend(df))
        if len(windows) == 0:
            return {"status": "skipped", "reason": "no complete 30-second windows"}

        labels = windows["label"]
        target = labels.shift(-FORECAST_HORIZON)             # the next window's label
        has_target = target.notna().to_numpy()
        y_rows = windows.loc[has_target, ["start_s", "end_s"]].copy()
        y_rows["c1_target"] = target[has_target].to_numpy()
    except Exception as exc:
        return {"status": "skipped", "reason": f"processing error: {type(exc).__name__}: {exc}"}

    # Model input: 30 HR + 30 speed + 30 cadence + 30 temperature + distance at the window start
    x_rows, has_input, x_error = [], np.zeros(len(y_rows), dtype=bool), None
    try:
        cadence, temperature, distance = df["cadence"].to_numpy(), df["temperature"].to_numpy(), df["distance"].to_numpy()
        for k, (start, end) in enumerate(zip(y_rows["start_s"].to_numpy(), y_rows["end_s"].to_numpy())):
            start, end = int(start), int(end)
            if end > len(heart_rate):
                continue
            w_hr, w_speed, w_cad, w_temp = heart_rate[start:end], speed[start:end], cadence[start:end], temperature[start:end]
            if any(len(a) != WINDOW_SECONDS for a in (w_hr, w_speed, w_cad, w_temp)):
                continue
            if np.isnan(w_hr).any() or np.isnan(w_speed).any():
                continue
            x_rows.append(np.concatenate([w_hr, w_speed,
                                          np.nan_to_num(w_cad, nan=0.0), np.nan_to_num(w_temp, nan=0.0),
                                          [np.nan_to_num(distance[start], nan=0.0)]]))
            has_input[k] = True
    except Exception as exc:
        x_rows, has_input, x_error = [], np.zeros(len(y_rows), dtype=bool), f"{type(exc).__name__}: {exc}"

    return {
        "status": "ok",                                                                       # the file was used
        "y_rows": y_rows,                                                                     # Y rows: start_s, end_s and c1_target for each window
        "x_rows": np.array(x_rows) if x_rows else np.empty((0, 4 * WINDOW_SECONDS + 1)),      # X rows: 121 numbers per window
        "has_input": has_input,                                                               # True for each Y row that also got an X row
        "x_error": x_error,                                                                   # None, or why the X rows failed
        "n_windows": len(windows),                                                            # how many 30-second windows the run was cut into
        "n_unlabelled": int(labels.isna().sum()),                                             # windows with no label (the warm-up)
        "label_counts": labels.value_counts().to_dict(),                                      # windows per label (not used later)
    }


## 6. Process every run, then join and save

In [15]:
t_start = time.time()
y_parts, x_parts, skipped = [], [], []
n_windows = n_unlabelled = n_with_target = run_idx = 0
Y_COLUMNS = ["run_idx", "date", "file_path", "start_s", "end_s", "c1_target"]

for i, (fpath, date) in enumerate(dated, start=1):
    res = process_run(fpath)                        # turn this one file into its rows: windows, targets, model inputs

    if res["status"] != "ok":                       # the file could not be used: note why, and move on
        skipped.append({"file_path": fpath.name, "reason": res["reason"]})
    else:
        y_rows = res["y_rows"].assign(run_idx=run_idx, date=date, file_path=fpath.name)   # add the run number, date and file name
        y_parts.append(y_rows[Y_COLUMNS][res["has_input"]])                               # this run's Y rows
        if len(res["x_rows"]):
            x_parts.append(res["x_rows"])                                               # this run's X rows
        if res["x_error"]:
            skipped.append({"file_path": fpath.name, "reason": "model-input rows dropped: " + res["x_error"]})
        n_windows += res["n_windows"]
        n_unlabelled += res["n_unlabelled"]
        n_with_target += len(y_rows)
        run_idx += 1                                # the next usable run gets the next number

    if i % 50 == 0 or i == len(dated):
        print(f"{i}/{len(dated)} files | {(time.time() - t_start) / 60:.1f} min", flush=True)

# Join the runs into one table and one array, and check they match
y_df = pd.concat(y_parts, ignore_index=True)
X_flat = np.concatenate(x_parts, axis=0)
assert len(X_flat) == len(y_df), "X and Y row counts differ: the files would be misaligned"

# Sort X and Y together: file name, then window start (the order of the original files)
order = y_df.sort_values(["file_path", "start_s"], kind="stable").index.to_numpy()
y_df = y_df.loc[order].reset_index(drop=True)
X_flat = X_flat[order]

# Save
OUT_DIR.mkdir(parents=True, exist_ok=True)
np.save(X_OUT, X_flat)
y_df.to_csv(Y_OUT, index=False)

files_skipped = [s for s in skipped if not s["reason"].startswith("model-input")]
print(f"Runs used: {run_idx} | files skipped: {len(files_skipped)}")
print(f"Windows: {n_windows} | unlabelled warm-up: {n_unlabelled} | with a next window: {n_with_target}")
print(f"Model input: {X_flat.shape[0]} rows x {X_flat.shape[1]} columns")
print("Target balance:", y_df["c1_target"].value_counts().to_dict())
print("Saved:", X_OUT.name, "|", Y_OUT.name)
print(f"Time: {(time.time() - t_start) / 60:.1f} min")
if skipped:
    print("\nSkipped files and why:")
    for s in skipped:
        print(f"  {s['file_path']}: {s['reason']}")

50/1891 files | 0.2 min
100/1891 files | 0.3 min
150/1891 files | 0.4 min
200/1891 files | 0.6 min
250/1891 files | 0.8 min
300/1891 files | 0.9 min
350/1891 files | 1.2 min
400/1891 files | 1.4 min
450/1891 files | 1.7 min
500/1891 files | 2.0 min
550/1891 files | 2.3 min
600/1891 files | 2.6 min
650/1891 files | 2.8 min
700/1891 files | 3.3 min
750/1891 files | 3.9 min
800/1891 files | 4.5 min
850/1891 files | 4.9 min
900/1891 files | 5.5 min
950/1891 files | 6.1 min
1000/1891 files | 6.6 min
1050/1891 files | 6.9 min
1100/1891 files | 7.4 min
1150/1891 files | 7.8 min
1200/1891 files | 8.1 min
1250/1891 files | 8.5 min
1300/1891 files | 8.9 min
1350/1891 files | 9.2 min
1400/1891 files | 9.5 min
1450/1891 files | 9.9 min
1500/1891 files | 10.5 min
1550/1891 files | 11.3 min
1600/1891 files | 11.9 min
1650/1891 files | 12.6 min
1700/1891 files | 13.4 min
1750/1891 files | 14.2 min
1800/1891 files | 15.1 min
1850/1891 files | 15.9 min
1891/1891 files | 16.6 min
Runs used: 1875 | files

## 7. Compare with the originals

Identical files have identical checksums (MD5). This compares each file saved above with the original of the same name without the suffix.


In [16]:
import hashlib


def md5(path):
    return hashlib.md5(path.read_bytes()).hexdigest()


for new, original in [(Y_OUT, OUT_DIR / "y_labels_c1_v2.csv"), (X_OUT, OUT_DIR / "X_flat_c1_v2.npy")]:
    if not original.exists():
        print(f"{new.name}: no original to compare with")
    else:
        result = "IDENTICAL to" if md5(new) == md5(original) else "DIFFERENT from"
        print(f"{new.name}: {result} {original.name}")

y_labels_c1_v2_rebuild.csv: IDENTICAL to y_labels_c1_v2.csv
X_flat_c1_v2_rebuild.npy: IDENTICAL to X_flat_c1_v2.npy
